# Entraînement YOLO compost sur Colab (GPU)

**Règle d'or : le code se modifie dans le repo et se commit, JAMAIS dans ce notebook.**
Ce notebook ne fait qu'orchestrer : clone, install, données, scripts, sauvegarde.
Colab est en LECTURE SEULE vis-à-vis de git : on clone, aucune cellule ne
commit ni ne push — rien de ce qui se passe ici n'apparaît sur GitHub.

Prérequis :
- runtime GPU (Exécution > Modifier le type d'exécution > T4 GPU) ;
- un token GitHub personnel classique (scope `repo`) dans les Secrets Colab
  sous `GITHUB_TOKEN` — il ne sert qu'au clone ;
- un zip par dataset sur Drive, nommés `MyDrive/compost/dataset_raw_<nom>.zip`
  (ex. `dataset_raw_zerowaste.zip`, `dataset_raw_taco.zip`) — chacun contient
  `images/` + `labels/` + `groups.csv` (sortie de `import_dataset.py`).
  La liste des `<nom>` se renseigne dans la variable `DATASETS` de la cellule 5.

In [ ]:
# 1. Clone du repo (token lu depuis les Secrets Colab — utilisé uniquement pour cloner)
BRANCH = 'yolo'   # branche de travail ; mettre 'main' après fusion
from google.colab import userdata
token = userdata.get('GITHUB_TOKEN')
!rm -rf /content/repo   # repart propre si la cellule a déjà tourné (sinon le clone échoue : dossier non vide)
!git clone --depth 1 --branch {BRANCH} https://{token}@github.com/TSResearch-hub/Compost_Waste_Yolo.git /content/repo
# le code d'entraînement est le sous-dossier compost-yolo du repo
%cd /content/repo/compost-yolo

In [ ]:
# 2. Installation des dépendances
!pip install -q -e .

In [ ]:
# 3. (désactivée) Montage de Google Drive — désormais fait dans la cellule 5
# from google.colab import drive
# drive.mount('/content/drive')

In [ ]:
# 4. (désactivée) Copie/dézippage de ZeroWaste — désormais géré par la liste DATASETS (cellule 5)
# !cp /content/drive/MyDrive/compost/dataset_raw.zip /content/
# !unzip -q -o /content/dataset_raw.zip -d /content/dataset_raw

In [ ]:
# 5. Préparation des datasets (split par session ; ils s'ACCUMULENT dans /content/dataset)
# Liste des datasets à préparer ; chacun = MyDrive/compost/dataset_raw_<nom>.zip
# Le split par hash garantit qu'une image déjà préparée ne change jamais de split.
DATASETS = ['zerowaste', 'taco']

from google.colab import drive
drive.mount('/content/drive')

for name in DATASETS:
    !cp /content/drive/MyDrive/compost/dataset_raw_{name}.zip /content/
    !unzip -q -o /content/dataset_raw_{name}.zip -d /content/dataset_raw_{name}
    !python scripts/prepare_dataset.py --source /content/dataset_raw_{name} --output /content/dataset

In [ ]:
# 5b. Histogramme par dataset INDIVIDUEL (sur le dataset brut, avant fusion/split)
# Permet de voir ce que chaque dataset apporte par classe, avant qu'ils soient mélangés.
import yaml
from collections import Counter
from pathlib import Path
import matplotlib.pyplot as plt

names = yaml.safe_load(open('configs/data.yaml'))['names']
x = range(len(names))

for name in DATASETS:
    counts = Counter()
    for label_file in Path(f'/content/dataset_raw_{name}/labels').glob('*.txt'):
        for line in label_file.read_text().splitlines():
            if line.strip():
                counts[int(line.split()[0])] += 1
    plt.figure(figsize=(10, 3))
    plt.bar(list(x), [counts[j] for j in x])
    plt.yscale('log')  # échelle log : sans elle, les classes rares sont invisibles
    plt.xticks(list(x), names, rotation=20)
    plt.ylabel('instances (échelle log)')
    plt.title(f"Dataset « {name} » — instances par classe (brut)")
    plt.grid(axis='y', alpha=0.3); plt.tight_layout()
    plt.show()
    print(f"{name}: {sum(counts.values())} instances —",
          ", ".join(f"{names[j]}: {counts[j]}" for j in x))

In [ ]:
# 5c. Histogramme GLOBAL (tous datasets fusionnés) par classe et par split
import yaml
from collections import Counter
from pathlib import Path
import matplotlib.pyplot as plt

names = yaml.safe_load(open('configs/data.yaml'))['names']
splits = ['train', 'val', 'test']
counts = {s: Counter() for s in splits}
for s in splits:
    for label_file in Path(f'/content/dataset/labels/{s}').glob('*.txt'):
        for line in label_file.read_text().splitlines():
            if line.strip():
                counts[s][int(line.split()[0])] += 1

x = range(len(names))
width = 0.27
plt.figure(figsize=(10, 4))
for i, s in enumerate(splits):
    plt.bar([v + (i - 1) * width for v in x], [counts[s][j] for j in x], width, label=s)
plt.yscale('log')  # échelle log : sans elle, les classes rares sont invisibles
plt.xticks(list(x), names, rotation=20)
plt.ylabel('instances (échelle log)')
plt.title('Instances par classe et par split (tous datasets confondus)')
plt.legend(); plt.grid(axis='y', alpha=0.3); plt.tight_layout()
plt.show()
for s in splits:
    print(f"{s}: {sum(counts[s].values())} instances —",
          ", ".join(f"{names[j]}: {counts[s][j]}" for j in x))

In [ ]:
# 6. Entraînement (checkpoints sauvegardés sur Drive toutes les 10 epochs)
# Reprise après coupure : ajouter --resume /content/runs/train_xxx/weights/last.pt
!python scripts/train.py --data /content/dataset/data.yaml \
    --runs-dir /content/runs \
    --backup-dir /content/drive/MyDrive/compost/backups --backup-every 10

In [ ]:
# 7. Copie du run complet (poids + métriques) vers Drive
!mkdir -p /content/drive/MyDrive/compost/runs
!cp -r /content/runs/* /content/drive/MyDrive/compost/runs/
!ls /content/drive/MyDrive/compost/runs